# Step 5 — Feedback Loop Simulation

**Goal:** Simulate the end-to-end loop where analyst-reviewed corrections flow back into training and improve the next model version. Measure the accuracy delta per correction batch on a held-out test set, and log each corrected model as a nested child run of the baseline so the lineage is explicit in MLflow.

**The framing:** Step 4 detected drift. Steps 1–4's audit framework would have paged an analyst. In a real fintech, the analyst would review borderline / high-confidence-wrong decisions and confirm true outcomes. Those corrections become high-quality labels that get fed back into training. This notebook simulates that loop.

**Inputs:**
- The audited MLflow run from Step 3 (baseline model + preprocessor + feature names)
- `data/processed/application_train_engineered.parquet` (re-split into train / production-pool / test)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import mlflow

from sklearn.model_selection import train_test_split

from src.utils.config import load_config, get_paths
from src.utils.logging import get_logger
from src.utils.mlflow_helpers import configure_mlflow
from src.data.loader import load_engineered_train
from src.models.evaluate import compute_metrics

cfg = load_config()
paths = get_paths(cfg)
SEED = cfg['project']['random_seed']
TARGET = cfg['project']['target_col']
ID_COL = cfg['project']['id_col']
np.random.seed(SEED)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100

configure_mlflow(experiment='loan-default-baseline')

# Find the audited baseline run.
runs = mlflow.search_runs(
    experiment_names=['loan-default-baseline'],
    filter_string='attributes.status = "FINISHED" and tags.audited = "true"',
    order_by=['attributes.start_time DESC'],
    max_results=1,
)
assert not runs.empty, 'No audited baseline found — complete Steps 2-3 first.'
baseline_run_id = runs.iloc[0]['run_id']
print(f'Baseline run: {baseline_run_id}')

baseline_model = mlflow.xgboost.load_model(f'runs:/{baseline_run_id}/model')
preprocessor = joblib.load(paths.artifacts / 'preprocessor.joblib')
with open(paths.artifacts / 'feature_names.json') as fh:
    feature_names = json.load(fh)

# ── 3-way split: train (80%) / production-pool (10%) / test (10%) ──────
df_fe = load_engineered_train()
X_all = df_fe.drop(columns=[TARGET, ID_COL])
y_all = df_fe[TARGET].astype('int32')

# First split: 80% train, 20% holdout. Matches Step 2 exactly so the
# loaded baseline model's training set is identical.
X_train, X_hold, y_train, y_hold = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED,
)
# Second split: divide the 20% holdout 50/50 into production-pool + test.
X_prod, X_test, y_prod, y_test = train_test_split(
    X_hold, y_hold, test_size=0.50, stratify=y_hold, random_state=SEED,
)

for name, X_, y_ in [('train', X_train, y_train),
                      ('production-pool', X_prod, y_prod),
                      ('test', X_test, y_test)]:
    print(f'  {name:<16}: {X_.shape[0]:>6,} rows  ({y_.mean():.2%} positive)')

# Reset indices so positional indexing is clean throughout.
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_prod = X_prod.reset_index(drop=True);   y_prod = y_prod.reset_index(drop=True)
X_test = X_test.reset_index(drop=True);   y_test = y_test.reset_index(drop=True)

# Transform with the saved preprocessor (same as everywhere else in the project).
X_train_mat = preprocessor.transform(X_train)
X_prod_mat = preprocessor.transform(X_prod)
X_test_mat = preprocessor.transform(X_test)


# ── Baseline performance on the fresh test set (the "before" number) ───
def score_with(model, X_mat):
    """Score a matrix with a booster, using its (sliced) iteration range."""
    X_df = pd.DataFrame(X_mat, columns=feature_names)
    return model.predict(xgb.DMatrix(X_df))


baseline_test_scores = score_with(baseline_model, X_test_mat)
baseline_metrics = compute_metrics(
    y_test.values, baseline_test_scores,
    precision_targets=(0.50,),
)
print(f'\nBaseline performance on TEST set (never seen during training):')
print(f'  PR-AUC:           {baseline_metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC:          {baseline_metrics["roc_auc"]:.4f}')
print(f'  recall@p50:       {baseline_metrics["recall_at_p50"]:.4f}')

log = get_logger('step5')
log.info('Step 5 environment ready. Baseline test PR-AUC: %.4f', baseline_metrics['pr_auc'])

In [ ]:
from src.feedback.correction import find_high_confidence_errors

# Score the production pool with the baseline model.
prod_scores = score_with(baseline_model, X_prod_mat)
print(f'Production pool: {len(X_prod):,} applicants scored.')
print(f'Score percentiles:  '
      f'p50={np.percentile(prod_scores, 50):.3f}  '
      f'p90={np.percentile(prod_scores, 90):.3f}  '
      f'p99={np.percentile(prod_scores, 99):.3f}')

# Identify the two types of high-confidence errors.
fp_candidates, fn_candidates = find_high_confidence_errors(
    y_true=y_prod.values,
    y_score=prod_scores,
    fp_threshold=0.30,   # model loudly predicted default
    fn_threshold=0.03,   # model loudly predicted safe
)

print(f'\nFalse-positive candidates (model said default, applicant repaid):')
print(f'  n = {fp_candidates.n}, score range = '
      f'({fp_candidates.score_range[0]:.3f}, {fp_candidates.score_range[1]:.3f})')

print(f'\nFalse-negative candidates (model said safe, applicant defaulted):')
print(f'  n = {fn_candidates.n}, score range = '
      f'({fn_candidates.score_range[0]:.3f}, {fn_candidates.score_range[1]:.3f})')

# Visualize: where do these errors live on the score distribution?
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(prod_scores[y_prod.values == 0], bins=60, alpha=0.5, color='#4C72B0', label='true repaid')
ax.hist(prod_scores[y_prod.values == 1], bins=60, alpha=0.5, color='#C44E52', label='true defaulted')
ax.axvline(0.30, ls='--', color='red', lw=1, label='FP threshold (score >= 0.30)')
ax.axvline(0.03, ls='--', color='blue', lw=1, label='FN threshold (score <= 0.03)')
ax.set_yscale('log')
ax.set_xlabel('baseline predicted probability')
ax.set_ylabel('count (log scale)')
ax.set_title('Production pool: scores by true outcome, with correction thresholds')
ax.legend(fontsize=8, loc='upper right')
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
from src.feedback.correction import sample_correction_batch, build_augmented_training_set
from src.models.train import train_xgb, default_xgb_params

BATCH_SIZE = 100
CORRECTION_WEIGHT = 3.0
N_BATCHES = 3

# Track the trajectory across batches. Row 0 = baseline (no corrections).
trajectory = [{
    'batch': 0,
    'corrections_total': 0,
    'pr_auc': baseline_metrics['pr_auc'],
    'pr_auc_delta_vs_baseline': 0.0,
    'recall_at_p50': baseline_metrics['recall_at_p50'],
    'recall_delta_vs_baseline': 0.0,
    'roc_auc': baseline_metrics['roc_auc'],
    'best_iteration': None,
}]

for batch_num in range(1, N_BATCHES + 1):
    print(f'\n──────── Correction batch {batch_num} ────────')

    # Sample fresh correction indices for this batch
    correction_indices = sample_correction_batch(
        candidates=[fp_candidates, fn_candidates],
        batch_size=BATCH_SIZE,
        seed=SEED + batch_num,
    )
    print(f'Sampled {len(correction_indices)} corrections')

    # All corrections accumulated so far — each batch adds to the previous
    if batch_num == 1:
        accumulated_idx = correction_indices
    else:
        accumulated_idx = np.unique(np.concatenate([accumulated_idx, correction_indices]))
    print(f'Total accumulated corrections: {len(accumulated_idx)}')

    # Build the augmented training set
    X_aug, y_aug, weights = build_augmented_training_set(
        X_train_original=X_train,
        y_train_original=y_train,
        X_corrections=X_prod.iloc[accumulated_idx].reset_index(drop=True),
        y_corrections=y_prod.iloc[accumulated_idx].reset_index(drop=True),
        correction_weight=CORRECTION_WEIGHT,
    )

    # Transform with the same Step 1 preprocessor (still the single source of truth)
    X_aug_mat = preprocessor.transform(X_aug)

    # Carve a small (5%) validation slice off the augmented training set for early stopping
    from sklearn.model_selection import train_test_split as _tts
    X_tr, X_val, y_tr, y_val_es, w_tr, w_val = _tts(
        X_aug_mat, y_aug.values, weights,
        test_size=0.05, stratify=y_aug.values, random_state=SEED,
    )

    # Retrain XGBoost — sample weights ensure the correction rows pull harder on the loss
    result = train_xgb(
        X_train=X_tr, y_train=y_tr,
        X_val=X_val, y_val=y_val_es,
        params=default_xgb_params(),
        num_boost_round=600,
        early_stopping_rounds=30,
        feature_names=feature_names,
        sample_weight=w_tr,
    )

    # Slice to best iteration before evaluation + logging (the production pattern)
    retrained_model = result.model[: result.best_iteration + 1]

    # Evaluate on the same held-out test set as the baseline
    test_scores = score_with(retrained_model, X_test_mat)
    metrics = compute_metrics(y_test.values, test_scores, precision_targets=(0.50,))

    pr_auc_delta = metrics['pr_auc'] - baseline_metrics['pr_auc']
    recall_delta = metrics['recall_at_p50'] - baseline_metrics['recall_at_p50']

    print(f'  Test PR-AUC:       {metrics["pr_auc"]:.4f}  (Δ {pr_auc_delta:+.4f} vs baseline)')
    print(f'  Test recall@p50:   {metrics["recall_at_p50"]:.4f}  (Δ {recall_delta:+.4f} vs baseline)')

    # Log as a nested child of the baseline run — explicit lineage in MLflow
    with mlflow.start_run(run_id=baseline_run_id):
        with mlflow.start_run(run_name=f'corrected-v{batch_num}', nested=True) as child:
            mlflow.set_tags({
                'role': 'corrected',
                'parent_run_id': baseline_run_id,
                'correction_batch': str(batch_num),
                'correction_strategy': 'high-confidence-errors-balanced-FP-FN',
            })
            mlflow.log_params({
                'corrections_in_batch': BATCH_SIZE,
                'corrections_total': len(accumulated_idx),
                'correction_weight': CORRECTION_WEIGHT,
                'best_iteration': result.best_iteration,
            })
            mlflow.log_metrics({
                **metrics,
                'pr_auc_delta_vs_baseline': pr_auc_delta,
                'recall_at_p50_delta_vs_baseline': recall_delta,
            })
            mlflow.xgboost.log_model(
                xgb_model=retrained_model,
                name='model',
                input_example=pd.DataFrame(X_tr[:5], columns=feature_names),
            )

    trajectory.append({
        'batch': batch_num,
        'corrections_total': len(accumulated_idx),
        'pr_auc': metrics['pr_auc'],
        'pr_auc_delta_vs_baseline': pr_auc_delta,
        'recall_at_p50': metrics['recall_at_p50'],
        'recall_delta_vs_baseline': recall_delta,
        'roc_auc': metrics['roc_auc'],
        'best_iteration': result.best_iteration,
    })

trajectory_df = pd.DataFrame(trajectory)
print('\n══════════ Final trajectory ══════════')
display(trajectory_df.round(4))

In [ ]:
# Trajectory plot — PR-AUC and recall vs corrections accumulated.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PR-AUC trajectory
axes[0].plot(trajectory_df['corrections_total'], trajectory_df['pr_auc'],
             marker='o', lw=2, color='#4C72B0', label='retrained model')
axes[0].axhline(baseline_metrics['pr_auc'], ls='--', color='black', lw=1,
                label='baseline')
axes[0].set_xlabel('corrections accumulated')
axes[0].set_ylabel('test PR-AUC')
axes[0].set_title('PR-AUC vs correction batches')
axes[0].legend()
for spine in ('top', 'right'):
    axes[0].spines[spine].set_visible(False)

# Recall@p50 trajectory
axes[1].plot(trajectory_df['corrections_total'], trajectory_df['recall_at_p50'],
             marker='o', lw=2, color='#55A868', label='retrained model')
axes[1].axhline(baseline_metrics['recall_at_p50'], ls='--', color='black', lw=1,
                label='baseline')
axes[1].set_xlabel('corrections accumulated')
axes[1].set_ylabel('test recall @ 50% precision')
axes[1].set_title('Recall @ p50 vs correction batches')
axes[1].legend()
for spine in ('top', 'right'):
    axes[1].spines[spine].set_visible(False)

fig.tight_layout()
plt.show()

# The "promotion gate" — would we deploy any of these corrected models?
print('\nPromotion-gate decisions (would we replace baseline?):')
promotion_threshold = 0.005   # require at least +0.5 PR-AUC points to promote
for _, row in trajectory_df.iterrows():
    if row['batch'] == 0:
        continue
    decision = 'PROMOTE' if row['pr_auc_delta_vs_baseline'] >= promotion_threshold else 'DO NOT PROMOTE'
    print(f"  corrected-v{int(row['batch'])} ({int(row['corrections_total'])} corrections): "
          f"Δ PR-AUC = {row['pr_auc_delta_vs_baseline']:+.4f}  →  {decision}")

# Save the trajectory CSV alongside the other audit artifacts
trajectory_path = paths.artifacts / 'feedback_trajectory.csv'
trajectory_df.to_csv(trajectory_path, index=False)

# Attach the trajectory to the parent (baseline) MLflow run too — so it sits
# alongside the audit and the monitoring report.
with mlflow.start_run(run_id=baseline_run_id):
    mlflow.log_artifact(str(trajectory_path), artifact_path='feedback')
    mlflow.log_figure(fig, 'feedback/correction_trajectory.png')
    mlflow.set_tag('feedback_loop_run', 'true')
    mlflow.log_metrics({
        'feedback_best_pr_auc_delta': float(trajectory_df['pr_auc_delta_vs_baseline'].max()),
        'feedback_n_promoted': int((trajectory_df['pr_auc_delta_vs_baseline'] >= promotion_threshold).sum()),
        'feedback_corrections_evaluated': int(trajectory_df['corrections_total'].max()),
    })

print(f'\nTrajectory and plot logged to baseline run.')

## Step 5 — Summary

Built the closing piece of the trust-infrastructure stack — a closed-loop retraining pipeline that takes analyst-verified corrections and decides whether to promote the corrected model to production. Three-way split (train / production-pool / test) so corrections are sampled from one held-out set and evaluation happens on another, never-seen set. Each of three retraining runs is logged as a nested MLflow child of the baseline, so the lineage `baseline → corrected-v1 → corrected-v2 → corrected-v3` is queryable.

**Headline finding — the framework correctly blocked promotion.** Adding 100, 183, 246 high-confidence-error corrections with 3× sample weight slightly *degraded* test PR-AUC (Δ -0.0025, -0.0037, -0.0016). None of the corrected versions cleared the promotion gate. The baseline stays in production while the data science team revisits the strategy.

**Why this is the right result:**

- Naive intuition says "more labeled data = better model." Reality: the corrections sampled were the model's *hardest* cases (extreme false positives and false negatives), and 3× weighting them caused the loss to focus disproportionately on those few rows. Generalization suffered.
- A real fintech without this framework would have shipped the corrected model on the assumption that "analyst corrections must help." Months later they'd notice the model's performance had quietly degraded. The promotion gate is what prevents that.

**Three calls worth flagging:**

1. **Test set never sees corrections.** Baseline and retrained both evaluate on the same 30K held-out rows. Any other comparison would be confounded.
2. **Nested child runs in MLflow.** Lineage is explicit — a future reviewer can trace any corrected model back to its parent and see exactly which corrections went in.
3. **Promotion gate is a real metric threshold (+0.005 PR-AUC), not a vibe.** Reviewers know in advance what "better" means; the gate makes the deploy/don't-deploy decision auditable.

**Next steps a real team would take (not in scope for this notebook):**

- Sample borderline errors (scores near the operating threshold), not extreme ones
- Drop the sample weight from 3× to 1.5× or just 1×
- Wait for larger correction batches — 100 rows on a 246K training set is < 0.05% of training data, not enough signal to expect improvement

**Handoff to Step 6 (Streamlit dashboard):** the audited model run, the corrected child runs (for showing version history), the SHAP explainer (for per-decision reason codes on the dashboard's selected applicant), and the drift + feedback tables (for the monitoring panel).